# Notebook 22. Objective Coastal-Box Metrics and Provisional Regime Labels

This notebook starts the **simpler, physically focused classification** requested after the EOF/clustering work became hard to interpret for the main science question.

The goal here is **not** to build a final manual label set yet. The goal is to:

- define one primary **Sea-of-Japan coastal strip** polygon and keep the existing **JPCZ polygon**
- compute the four raw event-level means for all 201 events:
  - JPCZ-polygon mean `850 hPa q×(-ω)` moisture-flux proxy
  - coastal-strip mean `850 hPa q×(-ω)` moisture-flux proxy
  - JPCZ-polygon mean `925 hPa` signed divergence
  - coastal-strip mean `925 hPa` signed divergence
- test **candidate thresholds** objectively
- assign **provisional regime labels** using only those raw means
- show low-level composites for the provisional offshore-JPCZ and coastal-interaction groups
- keep the old cleaned cluster labels only as **reference columns at the end**, not as the main method

The advisor's logic translated into this notebook is:

- focus on the two most physically direct proxies: **moisture flux** and **divergence/convergence**
- use an **objective coastal-box rule first**
- use **tested thresholds** rather than relying on the older clusters alone
- save the resulting event table so the next notebook can do **manual verification and timestamping**

Important note:

- This notebook keeps the raw means in their original sign convention.
- `925 hPa` divergence stays as **signed divergence**, so **more negative values still mean stronger convergence**.
- Real zeros are included in the means; only missing/NaN cells are excluded.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, OBJECTIVE_SUBTYPE_DOMAIN
from jpcz_catalog.detect import prepare_detection_geometry

CLEANED_RUN_EXPORT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
EOF_EXPORT_DIR = Path("outputs/verification/objective_subtype_eof_analysis")
OBJECTIVE_EXPORT_DIR = Path("outputs/verification/objective_coastal_box_regimes")
OBJECTIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_coastal_box_regime_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CLUSTERED_EVENTS_PATH = CLEANED_RUN_EXPORT_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"
LOW_LEVEL_STACK_PATH = EOF_EXPORT_DIR / "cleaned_low_level_eof_event_fields.nc"

PRIMARY_CLUSTER_COLUMN = "cleaned_cluster_ward_2"
SECONDARY_CLUSTER_COLUMN = "cleaned_cluster_ward_3"

METRIC_FIELDS = [
    "vertical_moisture_flux_proxy_850_peak",
    "divergence_925_peak",
]
REFERENCE_FIELDS = [
    "z850_anomaly_min_tminus12_to_tplus12",
]
COMPOSITE_FIELDS = METRIC_FIELDS
PLOT_DOMAIN = OBJECTIVE_SUBTYPE_DOMAIN

# First-pass diagonal coastal strip polygon parallel to the Sea-of-Japan-facing
# coastline of Japan. This is intentionally editable and should be refined if
# the manual review indicates a mismatch.
COASTAL_STRIP_VERTICES = (
    (129.25, 36.25),
    (139.75, 44.75),
    (141.75, 42.75),
    (131.25, 34.25),
)

COASTAL_QFLUX_SPLIT_QUANTILES = (0.60, 0.65, 0.70, 0.75, 0.80)
POLYGON_QFLUX_MIN_QUANTILES = (0.55, 0.60, 0.65, 0.70)
POLYGON_DIV_MAX_QUANTILES = (0.20, 0.25, 0.30, 0.35)
COASTAL_DIV_MAX_QUANTILES = (0.20, 0.25, 0.30, 0.35)

DEFAULT_COASTAL_QFLUX_SPLIT_Q = 0.75
DEFAULT_POLYGON_QFLUX_MIN_Q = 0.60
DEFAULT_POLYGON_DIV_MAX_Q = 0.25
DEFAULT_COASTAL_DIV_MAX_Q = 0.25

FIELD_DISPLAY_NAMES = {
    "vertical_moisture_flux_proxy_850_peak": "850 hPa vertical moisture-flux proxy at event peak (Russian coastal exclusion)",
    "divergence_925_peak": "925 hPa signed divergence at event peak (Russian coastal exclusion)",
    "z850_anomaly_min_tminus12_to_tplus12": "850 hPa z anomaly minimum over t-12/t0/t+12 (Russian coastal exclusion)",
}
SHORT_FIELD_DISPLAY_NAMES = {
    "vertical_moisture_flux_proxy_850_peak": "850 hPa q×(-ω) proxy",
    "divergence_925_peak": "925 hPa signed divergence",
    "z850_anomaly_min_tminus12_to_tplus12": "z850 anomaly",
}
FIELD_UNITS = {
    "vertical_moisture_flux_proxy_850_peak": "1e-3 Pa s^-1",
    "divergence_925_peak": "1e-5 s^-1",
    "z850_anomaly_min_tminus12_to_tplus12": "gpm",
}
FIELD_CMAPS = {
    "vertical_moisture_flux_proxy_850_peak": "BrBG",
    "divergence_925_peak": "RdBu_r",
    "z850_anomaly_min_tminus12_to_tplus12": "RdBu_r",
}
LABEL_COLORS = {
    "offshore_jpcz_core": "#1b9e77",
    "coastal_interaction": "#d95f02",
    "mixed_transition": "#7570b3",
    "weak_or_unclear": "#7f7f7f",
}

EVENT_METRICS_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_event_metrics.csv"
THRESHOLD_TABLE_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_default_thresholds.csv"
THRESHOLD_SENSITIVITY_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_threshold_sensitivity.csv"
LABEL_COUNT_SUMMARY_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_label_counts.csv"
OBJECTIVE_K2_CROSSTAB_PATH = OBJECTIVE_EXPORT_DIR / "objective_vs_cleaned_k2_counts.csv"
OBJECTIVE_K3_CROSSTAB_PATH = OBJECTIVE_EXPORT_DIR / "objective_vs_cleaned_k3_counts.csv"
GROUP_COMPOSITES_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_group_composites.nc"
GROUP_SAMPLE_COUNT_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_group_sample_counts.csv"
GROUP_BOX_SUMMARY_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_group_box_means.csv"
PLOT_INVENTORY_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_plot_inventory.csv"
PLOT_LEVEL_SUMMARY_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_plot_level_summary.csv"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None



def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive cache:", drive_path)
    return True



def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()



def size_rank_descriptor(rank: int, total: int) -> str:
    if rank == 1:
        return "largest subgroup"
    if rank == total:
        return "smallest subgroup"
    return f"size rank {rank} of {total}"



def build_cluster_labels_from_counts(cluster_counts: pd.Series | dict[int, int]):
    counts_dict = {int(cluster_id): int(n_events) for cluster_id, n_events in dict(cluster_counts).items()}
    ranked = sorted(counts_dict.items(), key=lambda item: (-item[1], item[0]))
    rank_lookup = {cluster_id: rank for rank, (cluster_id, _) in enumerate(ranked, start=1)}
    total = len(ranked)
    label_lookup = {}
    rows = []
    for cluster_id, n_events in sorted(counts_dict.items()):
        descriptor = size_rank_descriptor(rank_lookup[cluster_id], total)
        label_lookup[cluster_id] = f"Cluster {cluster_id}: n={n_events} ({descriptor})"
        rows.append(
            {
                "cluster_id": cluster_id,
                "n_events": n_events,
                "cluster_label": label_lookup[cluster_id],
                "size_rank": rank_lookup[cluster_id],
                "size_descriptor": descriptor,
            }
        )
    return label_lookup, pd.DataFrame(rows)



def lat_weighted_field_mean(field: xr.DataArray, weight_field: xr.DataArray) -> xr.DataArray:
    valid = xr.apply_ufunc(np.isfinite, field)
    weighted_field = field.where(valid, 0.0) * weight_field.where(valid, 0.0)
    denominator = weight_field.where(valid, 0.0).sum(dim=("latitude", "longitude"))
    numerator = weighted_field.sum(dim=("latitude", "longitude"))
    return xr.where(denominator > 0.0, numerator / denominator, np.nan)



def coordinate_edges(coord_values: np.ndarray) -> np.ndarray:
    coord_values = np.asarray(coord_values, dtype=float)
    if coord_values.ndim != 1:
        raise ValueError("Expected a 1D coordinate array when building pcolormesh edges")
    if coord_values.size == 1:
        half_step = 0.125
        return np.array([coord_values[0] - half_step, coord_values[0] + half_step], dtype=float)
    diffs = np.diff(coord_values)
    edges = np.empty(coord_values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (coord_values[:-1] + coord_values[1:])
    edges[0] = coord_values[0] - 0.5 * diffs[0]
    edges[-1] = coord_values[-1] + 0.5 * diffs[-1]
    return edges



def draw_scalar_pcolormesh(ax, field: xr.DataArray, *, cmap_name: str, levels: np.ndarray, extend: str = "both"):
    lon_edges = coordinate_edges(field.longitude.values)
    lat_edges = coordinate_edges(field.latitude.values)
    n_level_bins = max(len(levels) - 1, 1)
    n_extend_bins = {"neither": 0, "min": 1, "max": 1, "both": 2}.get(extend, 0)
    cmap = plt.get_cmap(cmap_name).resampled(n_level_bins + n_extend_bins)
    norm = mcolors.BoundaryNorm(levels, ncolors=cmap.N, clip=False, extend=extend)
    return ax.pcolormesh(
        lon_edges,
        lat_edges,
        field.values,
        cmap=cmap,
        norm=norm,
        shading="flat",
        transform=ccrs.PlateCarree(),
        linewidth=0.0,
        antialiased=False,
        snap=True,
    )



def add_map_features(ax, domain, *, title: str):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="50m", linewidth=0.9)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
    ax.add_feature(cfeature.LAND, facecolor="#f2f2f2", alpha=0.6)
    gl = ax.gridlines(draw_labels=True, linewidth=0.25, alpha=0.35)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title, fontsize=11)
    return gl



def overlay_polygon(ax, vertices, *, color: str, linewidth: float = 2.0, linestyle: str = "-"):
    lon_vals = [vertex[0] for vertex in vertices] + [vertices[0][0]]
    lat_vals = [vertex[1] for vertex in vertices] + [vertices[0][1]]
    ax.plot(lon_vals, lat_vals, color=color, linewidth=linewidth, linestyle=linestyle, transform=ccrs.PlateCarree())



def finite_field_values(*fields: xr.DataArray) -> np.ndarray:
    arrays = []
    for field in fields:
        values = np.asarray(field.values, dtype=float)
        finite = values[np.isfinite(values)]
        if finite.size > 0:
            arrays.append(finite)
    if not arrays:
        return np.array([0.0])
    return np.concatenate(arrays)



def robust_abs_limit(values: np.ndarray, *, upper_q: float = 0.99) -> float:
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 1.0
    return float(np.nanquantile(np.abs(finite), upper_q))



def rounded_limit(limit: float, step: float, *, minimum: float) -> float:
    return max(minimum, step * np.ceil(max(limit, 0.0) / step))



def build_composite_levels(field_name: str, offshore_field: xr.DataArray, coastal_field: xr.DataArray, diff_field: xr.DataArray) -> np.ndarray:
    values = finite_field_values(offshore_field, coastal_field, diff_field)
    if field_name == "divergence_925_peak":
        step = 1.0
        limit = rounded_limit(robust_abs_limit(values, upper_q=0.99), step, minimum=5.0)
        return np.arange(-limit, limit + step * 0.5, step)
    if field_name == "vertical_moisture_flux_proxy_850_peak":
        step = 0.05
        limit = rounded_limit(robust_abs_limit(values, upper_q=0.99), step, minimum=0.60)
        return np.arange(-limit, limit + step * 0.5, step)
    step = 10.0
    limit = rounded_limit(robust_abs_limit(values, upper_q=0.99), step, minimum=80.0)
    return np.arange(-limit, limit + step * 0.5, step)



def summarize_level_range(field_name: str, offshore_field: xr.DataArray, coastal_field: xr.DataArray, diff_field: xr.DataArray, levels: np.ndarray) -> dict:
    values = finite_field_values(offshore_field, coastal_field, diff_field)
    return {
        "field": field_name,
        "data_min": float(np.min(values)),
        "data_max": float(np.max(values)),
        "level_min": float(levels[0]),
        "level_max": float(levels[-1]),
        "level_step": float(levels[1] - levels[0]) if len(levels) > 1 else float("nan"),
        "n_bins": int(max(len(levels) - 1, 0)),
    }



def mean_dataset_over_events(ds: xr.Dataset, event_indices: list[int]) -> xr.Dataset:
    subset = ds.sel(event_index=event_indices).sortby("event_index")
    return subset.mean(dim="event_index", skipna=True)



def count_dataset_over_events(ds: xr.Dataset, event_indices: list[int]) -> xr.Dataset:
    subset = ds.sel(event_index=event_indices).sortby("event_index")
    rows = {}
    for field_name, field in subset.data_vars.items():
        rows[field_name] = xr.apply_ufunc(np.isfinite, field).sum(dim="event_index")
    return xr.Dataset(rows)



def assign_objective_labels(
    metrics_df: pd.DataFrame,
    *,
    polygon_qflux_min: float,
    polygon_div_max: float,
    coastal_qflux_split: float,
    coastal_div_max: float,
) -> pd.DataFrame:
    labeled_df = metrics_df.copy()
    offshore_mask = (
        np.isfinite(labeled_df["polygon_qflux_850_mean"])
        & np.isfinite(labeled_df["polygon_div_925_mean"])
        & np.isfinite(labeled_df["coastal_qflux_850_mean"])
        & (labeled_df["polygon_qflux_850_mean"] >= polygon_qflux_min)
        & (labeled_df["polygon_div_925_mean"] <= polygon_div_max)
        & (labeled_df["coastal_qflux_850_mean"] < coastal_qflux_split)
    )
    coastal_mask = (
        np.isfinite(labeled_df["coastal_qflux_850_mean"])
        & np.isfinite(labeled_df["coastal_div_925_mean"])
        & (labeled_df["coastal_qflux_850_mean"] >= coastal_qflux_split)
        & (labeled_df["coastal_div_925_mean"] <= coastal_div_max)
    )
    labels = np.full(len(labeled_df), "weak_or_unclear", dtype=object)
    labels[offshore_mask.values] = "offshore_jpcz_core"
    labels[coastal_mask.values] = "coastal_interaction"
    labels[(offshore_mask & coastal_mask).values] = "mixed_transition"
    labeled_df["offshore_rule_pass"] = offshore_mask.astype(bool)
    labeled_df["coastal_rule_pass"] = coastal_mask.astype(bool)
    labeled_df["objective_regime_label"] = labels
    return labeled_df



def summarize_label_counts(labeled_df: pd.DataFrame) -> pd.DataFrame:
    counts = labeled_df["objective_regime_label"].value_counts(dropna=False).rename_axis("objective_regime_label").reset_index(name="n_events")
    counts["fraction_of_201"] = (counts["n_events"] / len(labeled_df)).round(3)
    return counts



def plot_region_definition(domain, *, polygon_vertices, coastal_vertices):
    fig, ax = plt.subplots(figsize=(8.2, 6.0), subplot_kw={"projection": ccrs.PlateCarree()})
    add_map_features(ax, domain, title="Objective regions for the simplified regime method")
    overlay_polygon(ax, polygon_vertices, color="#1f78b4", linewidth=2.4)
    overlay_polygon(ax, coastal_vertices, color="#d95f02", linewidth=2.4)
    ax.text(129.2, 41.5, "JPCZ polygon", color="#1f78b4", fontsize=9.5, transform=ccrs.PlateCarree())
    ax.text(132.2, 35.2, "Coastal strip", color="#d95f02", fontsize=9.5, transform=ccrs.PlateCarree())
    fig.tight_layout()
    return fig



def plot_metric_distribution_grid(metrics_df: pd.DataFrame, threshold_lookup: dict):
    fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.0))
    panels = [
        ("polygon_qflux_850_mean", "Polygon mean 850 hPa q×(-ω)", threshold_lookup["polygon_qflux_min"]),
        ("coastal_qflux_850_mean", "Coastal-strip mean 850 hPa q×(-ω)", threshold_lookup["coastal_qflux_split"]),
        ("polygon_div_925_mean", "Polygon mean 925 hPa divergence", threshold_lookup["polygon_div_max"]),
        ("coastal_div_925_mean", "Coastal-strip mean 925 hPa divergence", threshold_lookup["coastal_div_max"]),
    ]
    for ax, (column, title, threshold) in zip(axes.flat, panels):
        values = metrics_df[column].to_numpy(dtype=float)
        finite = values[np.isfinite(values)]
        ax.hist(finite, bins=22, color="#bdbdbd", edgecolor="white")
        ax.axvline(threshold, color="#d95f02", linestyle="--", linewidth=2)
        ax.set_title(title)
        ax.set_xlabel(column)
        ax.set_ylabel("Event count")
    fig.tight_layout()
    return fig



def plot_metric_scatter_grid(metrics_df: pd.DataFrame, threshold_lookup: dict):
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.6))
    label_order = ["offshore_jpcz_core", "coastal_interaction", "mixed_transition", "weak_or_unclear"]

    for label in label_order:
        subset = metrics_df.loc[metrics_df["objective_regime_label"] == label]
        axes[0].scatter(
            subset["coastal_qflux_850_mean"],
            subset["polygon_qflux_850_mean"],
            s=28,
            alpha=0.8,
            color=LABEL_COLORS[label],
            label=label,
        )
        axes[1].scatter(
            subset["coastal_div_925_mean"],
            subset["polygon_div_925_mean"],
            s=28,
            alpha=0.8,
            color=LABEL_COLORS[label],
            label=label,
        )

    axes[0].axvline(threshold_lookup["coastal_qflux_split"], color="#d95f02", linestyle="--", linewidth=1.8)
    axes[0].axhline(threshold_lookup["polygon_qflux_min"], color="#1b9e77", linestyle="--", linewidth=1.8)
    axes[0].set_title("Moisture-flux regime space")
    axes[0].set_xlabel("Coastal-strip mean 850 hPa q×(-ω)")
    axes[0].set_ylabel("JPCZ-polygon mean 850 hPa q×(-ω)")

    axes[1].axvline(threshold_lookup["coastal_div_max"], color="#d95f02", linestyle="--", linewidth=1.8)
    axes[1].axhline(threshold_lookup["polygon_div_max"], color="#1b9e77", linestyle="--", linewidth=1.8)
    axes[1].set_title("Divergence regime space")
    axes[1].set_xlabel("Coastal-strip mean 925 hPa divergence")
    axes[1].set_ylabel("JPCZ-polygon mean 925 hPa divergence")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    return fig



def plot_regime_triptych(offshore_ds: xr.Dataset, coastal_ds: xr.Dataset, diff_ds: xr.Dataset, field_name: str, domain, *, levels: np.ndarray | None = None):
    offshore_field = offshore_ds[field_name]
    coastal_field = coastal_ds[field_name]
    diff_field = diff_ds[field_name]
    if levels is None:
        levels = build_composite_levels(field_name, offshore_field, coastal_field, diff_field)
    cmap_name = FIELD_CMAPS.get(field_name, "RdBu_r")
    fig, axes = plt.subplots(1, 3, figsize=(15.8, 5.8), subplot_kw={"projection": ccrs.PlateCarree()})
    panel_specs = [
        ("Offshore JPCZ", offshore_field),
        ("Coastal interaction", coastal_field),
        ("Coastal - Offshore", diff_field),
    ]
    cf_last = None
    for ax, (panel_title, field) in zip(axes, panel_specs):
        cf = draw_scalar_pcolormesh(ax, field, cmap_name=cmap_name, levels=levels, extend="both")
        cf_last = cf
        add_map_features(ax, domain, title=panel_title)
        overlay_polygon(ax, JPCZ_POLYGON_VERTICES, color="#1f78b4", linewidth=1.5)
        overlay_polygon(ax, COASTAL_STRIP_VERTICES, color="#d95f02", linewidth=1.5)
    fig.suptitle(FIELD_DISPLAY_NAMES[field_name], fontsize=12.5, y=0.97)
    fig.subplots_adjust(top=0.84, bottom=0.24, left=0.05, right=0.98, wspace=0.08)
    cax = fig.add_axes([0.18, 0.10, 0.64, 0.04])
    cbar = fig.colorbar(cf_last, cax=cax, orientation="horizontal")
    cbar.set_label(f"{SHORT_FIELD_DISPLAY_NAMES[field_name]} [{FIELD_UNITS[field_name]}]", labelpad=4)
    return fig


In [ ]:
paths_to_restore = [CLEANED_CLUSTERED_EVENTS_PATH, LOW_LEVEL_STACK_PATH]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

if not CLEANED_CLUSTERED_EVENTS_PATH.exists():
    raise RuntimeError("Run Notebook 15 first so the cleaned clustered event table exists.")
if not LOW_LEVEL_STACK_PATH.exists():
    raise RuntimeError("Run Notebook 17 first so the cleaned low-level event-field stack exists.")

clustered_df = pd.read_csv(CLEANED_CLUSTERED_EVENTS_PATH)
clustered_df = add_japan_local_time_columns(clustered_df)
required_columns = [PRIMARY_CLUSTER_COLUMN, SECONDARY_CLUSTER_COLUMN, "event_peak"]
missing_columns = [column for column in required_columns if column not in clustered_df.columns]
if missing_columns:
    raise RuntimeError(f"Missing required columns in the cleaned clustered event table: {missing_columns}")

low_level_stack_ds = load_cached_dataset(LOW_LEVEL_STACK_PATH)
if low_level_stack_ds is None:
    raise RuntimeError("Unable to restore the cleaned low-level event stack.")
missing_fields = [field for field in METRIC_FIELDS + REFERENCE_FIELDS if field not in low_level_stack_ds.data_vars]
if missing_fields:
    raise RuntimeError(f"The cleaned low-level stack is missing required fields: {missing_fields}")

geometry_polygon = prepare_detection_geometry(
    low_level_stack_ds.longitude,
    low_level_stack_ds.latitude,
    JPCZ_POLYGON_VERTICES,
)
geometry_coastal = prepare_detection_geometry(
    low_level_stack_ds.longitude,
    low_level_stack_ds.latitude,
    COASTAL_STRIP_VERTICES,
)

polygon_qflux_da = lat_weighted_field_mean(low_level_stack_ds["vertical_moisture_flux_proxy_850_peak"], geometry_polygon.weights)
coastal_qflux_da = lat_weighted_field_mean(low_level_stack_ds["vertical_moisture_flux_proxy_850_peak"], geometry_coastal.weights)
polygon_div_da = lat_weighted_field_mean(low_level_stack_ds["divergence_925_peak"], geometry_polygon.weights)
coastal_div_da = lat_weighted_field_mean(low_level_stack_ds["divergence_925_peak"], geometry_coastal.weights)

metrics_df = pd.DataFrame(
    {
        "event_index": low_level_stack_ds["event_index"].values.astype(int),
        "event_peak": pd.to_datetime(low_level_stack_ds["event_peak"].values),
        "polygon_qflux_850_mean": polygon_qflux_da.values.astype(float),
        "coastal_qflux_850_mean": coastal_qflux_da.values.astype(float),
        "polygon_div_925_mean": polygon_div_da.values.astype(float),
        "coastal_div_925_mean": coastal_div_da.values.astype(float),
        PRIMARY_CLUSTER_COLUMN: low_level_stack_ds["cleaned_cluster_k2"].values.astype(int),
        SECONDARY_CLUSTER_COLUMN: low_level_stack_ds["cleaned_cluster_k3"].values.astype(int),
    }
)
metadata_columns = [column for column in ["event_peak_jst", "duration_hours", "cluster_label", "subtype_label"] if column in clustered_df.columns]
metrics_df = metrics_df.merge(
    clustered_df[[PRIMARY_CLUSTER_COLUMN, SECONDARY_CLUSTER_COLUMN, "event_peak", "event_peak_jst", "duration_hours"]],
    left_on="event_index",
    right_index=True,
    how="left",
    suffixes=("", "_from_table"),
)
metrics_df = metrics_df.drop(columns=[f"{PRIMARY_CLUSTER_COLUMN}_from_table", f"{SECONDARY_CLUSTER_COLUMN}_from_table"], errors="ignore")
metrics_df = metrics_df.sort_values("event_peak").reset_index(drop=True)

quantile_lookup = {
    "polygon_qflux": {q: float(np.nanquantile(metrics_df["polygon_qflux_850_mean"].values, q)) for q in POLYGON_QFLUX_MIN_QUANTILES},
    "coastal_qflux": {q: float(np.nanquantile(metrics_df["coastal_qflux_850_mean"].values, q)) for q in COASTAL_QFLUX_SPLIT_QUANTILES},
    "polygon_div": {q: float(np.nanquantile(metrics_df["polygon_div_925_mean"].values, q)) for q in POLYGON_DIV_MAX_QUANTILES},
    "coastal_div": {q: float(np.nanquantile(metrics_df["coastal_div_925_mean"].values, q)) for q in COASTAL_DIV_MAX_QUANTILES},
}

default_threshold_lookup = {
    "polygon_qflux_min": quantile_lookup["polygon_qflux"][DEFAULT_POLYGON_QFLUX_MIN_Q],
    "coastal_qflux_split": quantile_lookup["coastal_qflux"][DEFAULT_COASTAL_QFLUX_SPLIT_Q],
    "polygon_div_max": quantile_lookup["polygon_div"][DEFAULT_POLYGON_DIV_MAX_Q],
    "coastal_div_max": quantile_lookup["coastal_div"][DEFAULT_COASTAL_DIV_MAX_Q],
}
threshold_summary_df = pd.DataFrame(
    [
        {"metric": "polygon_qflux_850_mean", "quantile_level": DEFAULT_POLYGON_QFLUX_MIN_Q, "threshold_value": default_threshold_lookup["polygon_qflux_min"], "rule_role": "minimum polygon moisture-flux support for offshore JPCZ"},
        {"metric": "coastal_qflux_850_mean", "quantile_level": DEFAULT_COASTAL_QFLUX_SPLIT_Q, "threshold_value": default_threshold_lookup["coastal_qflux_split"], "rule_role": "coastal moisture-flux split between offshore and coastal regimes"},
        {"metric": "polygon_div_925_mean", "quantile_level": DEFAULT_POLYGON_DIV_MAX_Q, "threshold_value": default_threshold_lookup["polygon_div_max"], "rule_role": "maximum polygon divergence allowed for offshore JPCZ (more negative = stronger convergence)"},
        {"metric": "coastal_div_925_mean", "quantile_level": DEFAULT_COASTAL_DIV_MAX_Q, "threshold_value": default_threshold_lookup["coastal_div_max"], "rule_role": "maximum coastal divergence allowed for coastal-interaction regime (more negative = stronger convergence)"},
    ]
)
threshold_summary_df["threshold_value"] = threshold_summary_df["threshold_value"].round(6)

threshold_sensitivity_rows = []
for polygon_q_q in POLYGON_QFLUX_MIN_QUANTILES:
    for coastal_q_q in COASTAL_QFLUX_SPLIT_QUANTILES:
        for polygon_div_q in POLYGON_DIV_MAX_QUANTILES:
            for coastal_div_q in COASTAL_DIV_MAX_QUANTILES:
                temp_labeled_df = assign_objective_labels(
                    metrics_df,
                    polygon_qflux_min=quantile_lookup["polygon_qflux"][polygon_q_q],
                    polygon_div_max=quantile_lookup["polygon_div"][polygon_div_q],
                    coastal_qflux_split=quantile_lookup["coastal_qflux"][coastal_q_q],
                    coastal_div_max=quantile_lookup["coastal_div"][coastal_div_q],
                )
                counts = temp_labeled_df["objective_regime_label"].value_counts()
                threshold_sensitivity_rows.append(
                    {
                        "polygon_qflux_min_quantile": polygon_q_q,
                        "coastal_qflux_split_quantile": coastal_q_q,
                        "polygon_div_max_quantile": polygon_div_q,
                        "coastal_div_max_quantile": coastal_div_q,
                        "polygon_qflux_min_value": quantile_lookup["polygon_qflux"][polygon_q_q],
                        "coastal_qflux_split_value": quantile_lookup["coastal_qflux"][coastal_q_q],
                        "polygon_div_max_value": quantile_lookup["polygon_div"][polygon_div_q],
                        "coastal_div_max_value": quantile_lookup["coastal_div"][coastal_div_q],
                        "offshore_jpcz_core_count": int(counts.get("offshore_jpcz_core", 0)),
                        "coastal_interaction_count": int(counts.get("coastal_interaction", 0)),
                        "mixed_transition_count": int(counts.get("mixed_transition", 0)),
                        "weak_or_unclear_count": int(counts.get("weak_or_unclear", 0)),
                        "thresholded_count": int(counts.get("offshore_jpcz_core", 0) + counts.get("coastal_interaction", 0) + counts.get("mixed_transition", 0)),
                        "is_default_threshold_set": bool(
                            polygon_q_q == DEFAULT_POLYGON_QFLUX_MIN_Q
                            and coastal_q_q == DEFAULT_COASTAL_QFLUX_SPLIT_Q
                            and polygon_div_q == DEFAULT_POLYGON_DIV_MAX_Q
                            and coastal_div_q == DEFAULT_COASTAL_DIV_MAX_Q
                        ),
                    }
                )
threshold_sensitivity_df = pd.DataFrame(threshold_sensitivity_rows)
threshold_sensitivity_df = threshold_sensitivity_df.sort_values(
    ["is_default_threshold_set", "mixed_transition_count", "weak_or_unclear_count", "offshore_jpcz_core_count"],
    ascending=[False, True, True, False],
).reset_index(drop=True)

labeled_metrics_df = assign_objective_labels(
    metrics_df,
    polygon_qflux_min=default_threshold_lookup["polygon_qflux_min"],
    polygon_div_max=default_threshold_lookup["polygon_div_max"],
    coastal_qflux_split=default_threshold_lookup["coastal_qflux_split"],
    coastal_div_max=default_threshold_lookup["coastal_div_max"],
)
label_count_summary_df = summarize_label_counts(labeled_metrics_df)

cluster_counts = clustered_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).value_counts().sort_index()
cluster_label_lookup, cluster_label_df = build_cluster_labels_from_counts(cluster_counts)
objective_vs_k2_counts_df = pd.crosstab(labeled_metrics_df["objective_regime_label"], labeled_metrics_df[PRIMARY_CLUSTER_COLUMN])
objective_vs_k3_counts_df = pd.crosstab(labeled_metrics_df["objective_regime_label"], labeled_metrics_df[SECONDARY_CLUSTER_COLUMN])

labeled_metrics_df.to_csv(EVENT_METRICS_PATH, index=False)
threshold_summary_df.to_csv(THRESHOLD_TABLE_PATH, index=False)
threshold_sensitivity_df.to_csv(THRESHOLD_SENSITIVITY_PATH, index=False)
label_count_summary_df.to_csv(LABEL_COUNT_SUMMARY_PATH, index=False)
objective_vs_k2_counts_df.to_csv(OBJECTIVE_K2_CROSSTAB_PATH)
objective_vs_k3_counts_df.to_csv(OBJECTIVE_K3_CROSSTAB_PATH)
for path in [
    EVENT_METRICS_PATH,
    THRESHOLD_TABLE_PATH,
    THRESHOLD_SENSITIVITY_PATH,
    LABEL_COUNT_SUMMARY_PATH,
    OBJECTIVE_K2_CROSSTAB_PATH,
    OBJECTIVE_K3_CROSSTAB_PATH,
]:
    maybe_copy_to_drive(path)

print("Cluster labels available for reference only")
display(cluster_label_df)
print("\nDefault tested thresholds used for the provisional label pass")
display(threshold_summary_df)
print("\nObjective regime counts from the default threshold set")
display(label_count_summary_df)
print("\nThreshold sensitivity table (top rows, sorted with the default set first)")
display(threshold_sensitivity_df.head(20))
print("\nObjective labels versus cleaned k=2 reference labels")
display(objective_vs_k2_counts_df)
print("\nObjective labels versus cleaned k=3 reference labels")
display(objective_vs_k3_counts_df)
print("\nFirst 15 labeled events in chronological order")
display(labeled_metrics_df.head(15))


In [ ]:
required_globals = [
    "labeled_metrics_df",
    "default_threshold_lookup",
    "threshold_sensitivity_df",
    "plot_region_definition",
    "plot_metric_distribution_grid",
    "plot_metric_scatter_grid",
    "maybe_copy_to_drive",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 22 metric-table cell before the plotting cell. "
        f"Missing globals: {missing_globals}"
    )

plot_inventory_rows = []

fig_regions = plot_region_definition(
    PLOT_DOMAIN,
    polygon_vertices=JPCZ_POLYGON_VERTICES,
    coastal_vertices=COASTAL_STRIP_VERTICES,
)
region_plot_path = PLOT_DIR / "objective_coastal_box_region_definition.png"
fig_regions.savefig(region_plot_path, dpi=160, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(region_plot_path)
plot_inventory_rows.append({"plot_kind": "region_definition", "field": "coastal_box_vs_polygon", "local_path": str(region_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / region_plot_path.name)})

fig_hists = plot_metric_distribution_grid(labeled_metrics_df, default_threshold_lookup)
distribution_plot_path = PLOT_DIR / "objective_coastal_box_metric_distributions.png"
fig_hists.savefig(distribution_plot_path, dpi=160, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(distribution_plot_path)
plot_inventory_rows.append({"plot_kind": "histogram_grid", "field": "raw_metric_distributions", "local_path": str(distribution_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / distribution_plot_path.name)})

fig_scatter = plot_metric_scatter_grid(labeled_metrics_df, default_threshold_lookup)
scatter_plot_path = PLOT_DIR / "objective_coastal_box_metric_scatter.png"
fig_scatter.savefig(scatter_plot_path, dpi=160, bbox_inches="tight")
plt.show()
maybe_copy_to_drive(scatter_plot_path)
plot_inventory_rows.append({"plot_kind": "scatter_grid", "field": "raw_metric_scatter", "local_path": str(scatter_plot_path), "drive_path": str(Path(DRIVE_OUTPUT_DIR) / scatter_plot_path.name)})

plot_inventory_df = pd.DataFrame(plot_inventory_rows)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, index=False)
maybe_copy_to_drive(PLOT_INVENTORY_PATH)
print("Saved Notebook 22 objective-method plots")
display(plot_inventory_df)


In [ ]:
required_globals = [
    "labeled_metrics_df",
    "low_level_stack_ds",
    "geometry_polygon",
    "geometry_coastal",
    "mean_dataset_over_events",
    "count_dataset_over_events",
    "lat_weighted_field_mean",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 22 metric-table cell before the group-composite cell. "
        f"Missing globals: {missing_globals}"
    )

offshore_event_indices = labeled_metrics_df.loc[labeled_metrics_df["objective_regime_label"] == "offshore_jpcz_core", "event_index"].astype(int).tolist()
coastal_event_indices = labeled_metrics_df.loc[labeled_metrics_df["objective_regime_label"] == "coastal_interaction", "event_index"].astype(int).tolist()
mixed_event_indices = labeled_metrics_df.loc[labeled_metrics_df["objective_regime_label"] == "mixed_transition", "event_index"].astype(int).tolist()
weak_event_indices = labeled_metrics_df.loc[labeled_metrics_df["objective_regime_label"] == "weak_or_unclear", "event_index"].astype(int).tolist()

if len(offshore_event_indices) == 0 or len(coastal_event_indices) == 0:
    raise RuntimeError(
        "The default threshold set did not produce both offshore and coastal groups. "
        "Adjust the tested thresholds in the config/helper cell before building composites."
    )

offshore_ds = mean_dataset_over_events(low_level_stack_ds[COMPOSITE_FIELDS + REFERENCE_FIELDS], offshore_event_indices)
coastal_ds = mean_dataset_over_events(low_level_stack_ds[COMPOSITE_FIELDS + REFERENCE_FIELDS], coastal_event_indices)
diff_ds = coastal_ds - offshore_ds

offshore_counts_ds = count_dataset_over_events(low_level_stack_ds[COMPOSITE_FIELDS + REFERENCE_FIELDS], offshore_event_indices)
coastal_counts_ds = count_dataset_over_events(low_level_stack_ds[COMPOSITE_FIELDS + REFERENCE_FIELDS], coastal_event_indices)

group_composites_ds = xr.merge(
    [
        offshore_ds.expand_dims(group=["offshore_jpcz_core"]),
        coastal_ds.expand_dims(group=["coastal_interaction"]),
        diff_ds.expand_dims(group=["coastal_minus_offshore"]),
    ],
    compat="override",
    join="outer",
)
group_composites_ds.to_netcdf(GROUP_COMPOSITES_PATH)
maybe_copy_to_drive(GROUP_COMPOSITES_PATH)

group_sample_count_rows = []
for group_name, count_ds, n_events in [
    ("offshore_jpcz_core", offshore_counts_ds, len(offshore_event_indices)),
    ("coastal_interaction", coastal_counts_ds, len(coastal_event_indices)),
]:
    for field_name in count_ds.data_vars:
        counts = count_ds[field_name]
        group_sample_count_rows.append(
            {
                "objective_regime_label": group_name,
                "field": field_name,
                "field_label": FIELD_DISPLAY_NAMES[field_name],
                "units": FIELD_UNITS[field_name],
                "n_events_in_group": n_events,
                "min_gridcell_count": int(np.nanmin(counts.values)),
                "median_gridcell_count": float(np.nanmedian(counts.values)),
                "max_gridcell_count": int(np.nanmax(counts.values)),
            }
        )
group_sample_count_df = pd.DataFrame(group_sample_count_rows)
group_sample_count_df.to_csv(GROUP_SAMPLE_COUNT_PATH, index=False)
maybe_copy_to_drive(GROUP_SAMPLE_COUNT_PATH)

group_box_summary_rows = []
for group_name, group_ds in [("offshore_jpcz_core", offshore_ds), ("coastal_interaction", coastal_ds), ("coastal_minus_offshore", diff_ds)]:
    for field_name in COMPOSITE_FIELDS:
        field = group_ds[field_name]
        polygon_mean = float(lat_weighted_field_mean(field, geometry_polygon.weights).values)
        coastal_mean = float(lat_weighted_field_mean(field, geometry_coastal.weights).values)
        group_box_summary_rows.extend(
            [
                {
                    "objective_regime_label": group_name,
                    "field": field_name,
                    "field_label": FIELD_DISPLAY_NAMES[field_name],
                    "region": "JPCZ polygon",
                    "weighted_mean": polygon_mean,
                },
                {
                    "objective_regime_label": group_name,
                    "field": field_name,
                    "field_label": FIELD_DISPLAY_NAMES[field_name],
                    "region": "Coastal strip",
                    "weighted_mean": coastal_mean,
                },
            ]
        )
group_box_summary_df = pd.DataFrame(group_box_summary_rows)
group_box_summary_df["weighted_mean"] = group_box_summary_df["weighted_mean"].round(3)
group_box_summary_df.to_csv(GROUP_BOX_SUMMARY_PATH, index=False)
maybe_copy_to_drive(GROUP_BOX_SUMMARY_PATH)

print("Notebook 22 objective-group counts")
display(pd.DataFrame([
    {"objective_regime_label": "offshore_jpcz_core", "n_events": len(offshore_event_indices)},
    {"objective_regime_label": "coastal_interaction", "n_events": len(coastal_event_indices)},
    {"objective_regime_label": "mixed_transition", "n_events": len(mixed_event_indices)},
    {"objective_regime_label": "weak_or_unclear", "n_events": len(weak_event_indices)},
]))
print("\nObjective-group sample-count summary")
display(group_sample_count_df)
print("\nObjective-group polygon and coastal-strip means")
display(group_box_summary_df)


In [ ]:
required_globals = [
    "offshore_ds",
    "coastal_ds",
    "diff_ds",
    "plot_regime_triptych",
    "build_composite_levels",
    "summarize_level_range",
    "maybe_copy_to_drive",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 22 objective-group composite cell before the composite plotting cell. "
        f"Missing globals: {missing_globals}"
    )

plot_level_rows = []
plot_inventory_rows = []

for field_name in COMPOSITE_FIELDS:
    levels = build_composite_levels(field_name, offshore_ds[field_name], coastal_ds[field_name], diff_ds[field_name])
    plot_level_rows.append(
        summarize_level_range(field_name, offshore_ds[field_name], coastal_ds[field_name], diff_ds[field_name], levels)
    )
    fig = plot_regime_triptych(offshore_ds, coastal_ds, diff_ds, field_name, PLOT_DOMAIN, levels=levels)
    plot_path = PLOT_DIR / f"objective_coastal_box_{field_name}_regime_triptych.png"
    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    maybe_copy_to_drive(plot_path)
    plot_inventory_rows.append({
        "plot_kind": "objective_regime_triptych",
        "field": field_name,
        "local_path": str(plot_path),
        "drive_path": str(Path(DRIVE_OUTPUT_DIR) / plot_path.name),
    })

plot_level_summary_df = pd.DataFrame(plot_level_rows)
plot_level_summary_df.to_csv(PLOT_LEVEL_SUMMARY_PATH, index=False)
maybe_copy_to_drive(PLOT_LEVEL_SUMMARY_PATH)
plot_inventory_df = pd.DataFrame(plot_inventory_rows)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, mode="a", index=False, header=not PLOT_INVENTORY_PATH.exists())
print("Selected display ranges for Notebook 22 objective-group composites")
display(plot_level_summary_df)
print("\nSaved Notebook 22 objective-group composite plots")
display(plot_inventory_df)


In [ ]:
required_globals = [
    "threshold_summary_df",
    "label_count_summary_df",
    "threshold_sensitivity_df",
    "objective_vs_k2_counts_df",
    "objective_vs_k3_counts_df",
    "group_box_summary_df",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 22 earlier cells before the summary cell. "
        f"Missing globals: {missing_globals}"
    )

print("Notebook 22 objective-method summary")
print("- This notebook uses only the four raw means requested by the simplified method.")
print("- The default thresholds are first-pass tested thresholds, not final truth.")
print("- Old cleaned cluster labels are shown below only as reference diagnostics.")
print("\nDefault threshold table")
display(threshold_summary_df)
print("\nObjective regime counts")
display(label_count_summary_df)
print("\nThreshold sensitivity rows nearest the default setting")
display(threshold_sensitivity_df.head(12))
print("\nReference overlap with cleaned k=2")
display(objective_vs_k2_counts_df)
print("\nReference overlap with cleaned k=3")
display(objective_vs_k3_counts_df)
print("\nKey polygon/coastal means for the objective groups")
display(group_box_summary_df)
print("\nHow to read Notebook 22")
print("- offshore_jpcz_core = strong polygon moisture-flux support, polygon divergence negative enough, and coastal moisture flux not too large")
print("- coastal_interaction = coastal-strip moisture flux large enough and coastal divergence negative enough")
print("- mixed_transition = both conditions true at once")
print("- weak_or_unclear = neither condition strong enough under the default threshold set")
print("- Notebook 23 should now use the saved event table from Notebook 22 for manual verification and timestamping")
